In [2]:
# === 1. Configuration and Imports ===
MIN_RADIUS = 5    # Minimum expected ball radius in pixels (adjust as needed)
MAX_RADIUS = 20   # Maximum expected ball radius in pixels (adjust as needed)
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import scipy.io as sio
from itertools import permutations

# === 2. Calibration Utilities ===
def getcameraparameters(file, ref_index):
    matlab_file = sio.loadmat(file)
    for k, v in matlab_file.items():
        if isinstance(v, np.ndarray):
            key = k
            break
    struct = matlab_file[key]
    mtx = struct['IntrinsicMatrix'][0][0].T
    rad_dist = struct['RadialDistortion'][0][0]
    
    dist = np.array([[rad_dist[0,0], rad_dist[0,1], 0, 0, rad_dist[0,2]]])
    img_dim = struct['ImageSize'][0][0]
    height = img_dim[0,0]
    width = img_dim[0,1]
    T = struct['TranslationVectors'][0][0][ref_index].reshape((3,1))
    R_vec = struct['RotationVectors'][0][0][ref_index].reshape((3,1))
    R, _ = cv2.Rodrigues(R_vec)
    return mtx, dist, T, R, R_vec, width, height

def AdjustedK(K, video_width, video_height, calib_width, calib_height):
    s1 = np.array([[video_width/calib_width, 0, 0], [0, video_height/calib_height, 0], [0, 0, 1]])
    return np.dot(s1, K)

# === 3. Ball Detection (All Circles) ===
def detect_all_balls(image, min_dist=20, param2=28):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (9, 9), 2)
    circles = cv2.HoughCircles(
        blurred, cv2.HOUGH_GRADIENT, dp=1.2, minDist=min_dist,
        param1=100, param2=param2,
        minRadius=MIN_RADIUS, maxRadius=MAX_RADIUS
    )
    if circles is not None:
        return [tuple(map(int, c)) for c in np.round(circles[0, :]).astype("int")]
    return []

# === 4. Marker Detection (Optional) ===
# def detect_markers(hsv_img, color_name, circular_mask):
#     ranges = {
#         'red':   [((0, 70, 50), (10, 255, 255)), ((160, 70, 50), (179, 255, 255))],
#         'green': [((50, 60, 40), (85, 255, 255))],
#         'blue':  [((95, 60, 40), (135, 255, 255))]
#     }
#     masks = []
#     for lower, upper in ranges[color_name]:
#         mask = cv2.inRange(hsv_img, np.array(lower), np.array(upper))
#         mask = cv2.bitwise_and(mask, circular_mask)
#         masks.append(mask)
#     combined_mask = masks[0] if len(masks) == 1 else cv2.bitwise_or(masks[0], masks[1])
#     contours, _ = cv2.findContours(combined_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
#     markers = []
#     for cnt in contours:
#         area = cv2.contourArea(cnt)
#         if 5 < area < 300:
#             M = cv2.moments(cnt)
#             if M["m00"] != 0:
#                 cx = int(M["m10"] / M["m00"])
#                 cy = int(M["m01"] / M["m00"])
#                 markers.append((cx, cy))
#     return markers

# # === 5. All-Permutations Ball Matching by Reprojection Error ===
# def best_ball_assignment_by_reproj_error(pts1, pts2, P1, P2):
#     """
#     pts1, pts2: N×2 arrays of (x, y) for detections in camera 1 and camera 2
#     Returns: permutation array for pts2 to align with pts1, and the min total error
#     """
#     N = pts1.shape[0]
#     best_indices = None
#     min_total_error = np.inf

#     for perm_indices in permutations(range(N)):
#         pts2_perm = pts2[list(perm_indices)]
#         # Triangulate
#         pts_1 = pts1.T
#         pts_2 = pts2_perm.T
#         pts_4d = cv2.triangulatePoints(P1, P2, pts_1, pts_2)
#         # print(pts_4d)
#         pts_3d = (pts_4d[:3] / pts_4d[3]).T

#         # Project 3D back to both images
#         reproj1 = (P1 @ np.hstack((pts_3d, np.ones((N, 1)))).T).T
#         reproj1 = (reproj1[:, :2].T / reproj1[:, 2]).T
#         error1 = np.linalg.norm(pts1 - reproj1, axis=1)

#         reproj2 = (P2 @ np.hstack((pts_3d, np.ones((N, 1)))).T).T
#         reproj2 = (reproj2[:, :2].T / reproj2[:, 2]).T
#         error2 = np.linalg.norm(pts2_perm - reproj2, axis=1)

#         total_error = error1.sum() + error2.sum()
#         if total_error < min_total_error:
#             min_total_error = total_error
#             print("error", min_total_error)
#             best_indices = np.array(perm_indices)
#             print("best_indices", best_indices)
#     return best_indices, min_total_error

# === 6. Gauss-Newton Refinement & Error Analysis ===
def JacobianMatrix(P, X):
    x, y, z = X
    denominator = P[2,0]*x + P[2,1]*y + P[2,2]*z + P[2,3]
    J = np.zeros((2, 3))
    J[0,0] = (P[0,0]*denominator - P[2,0]*(P[0,0]*x + P[0,1]*y + P[0,2]*z + P[0,3])) / denominator**2
    J[0,1] = (P[0,1]*denominator - P[2,1]*(P[0,0]*x + P[0,1]*y + P[0,2]*z + P[0,3])) / denominator**2
    J[0,2] = (P[0,2]*denominator - P[2,2]*(P[0,0]*x + P[0,1]*y + P[0,2]*z + P[0,3])) / denominator**2
    J[1,0] = (P[1,0]*denominator - P[2,0]*(P[1,0]*x + P[1,1]*y + P[1,2]*z + P[1,3])) / denominator**2
    J[1,1] = (P[1,1]*denominator - P[2,1]*(P[1,0]*x + P[1,1]*y + P[1,2]*z + P[1,3])) / denominator**2
    J[1,2] = (P[1,2]*denominator - P[2,2]*(P[1,0]*x + P[1,1]*y + P[1,2]*z + P[1,3])) / denominator**2
    return J

def GaussNewtonRefinement(pt3d, corr1, corr2, P1, P2, max_iter=10, threshold=1e-6):
    refined_pts = []
    for i in range(len(pt3d)):
        X = pt3d[i].copy()
        error = np.inf
        iterations = 0
        while error > threshold and iterations < max_iter:
            x_proj1 = P1 @ np.append(X, 1)
            x_proj1 /= x_proj1[2]
            x_proj2 = P2 @ np.append(X, 1)
            x_proj2 /= x_proj2[2]
            res1 = corr1[i] - x_proj1[:2]
            res2 = corr2[i] - x_proj2[:2]
            residual = np.concatenate([res1, res2])
            J1 = JacobianMatrix(P1, X)
            J2 = JacobianMatrix(P2, X)
            J = np.vstack([J1, J2])
            try:
                delta = np.linalg.inv(J.T @ J) @ J.T @ residual
            except np.linalg.LinAlgError:
                break
            X += delta
            error = np.linalg.norm(delta)
            iterations += 1
        refined_pts.append(X)
    return np.array(refined_pts)

def ReProjectionError(objpoints, imgpoints, P):
    errors = []
    for X, x in zip(objpoints, imgpoints):
        x_proj = P @ np.append(X, 1)
        x_proj /= x_proj[2]
        errors.append(np.linalg.norm(x - x_proj[:2]))
    return np.array(errors)

# === 7. Process Frames for Each Camera and Save ===
def process_frames_for_camera(frame_dir, mtx, dist, width, height):
    frame_files = sorted(list(Path(frame_dir).glob("*.jpg")))
    positions = []
    marker_records = []

    cropped_folder = Path(frame_dir) / "cropped_images"
    # annotated_folder = Path(frame_dir) / "annotated_images"
    debug_folder = Path(frame_dir) / "debug_detections"
    cropped_folder.mkdir(parents=True, exist_ok=True)
    # annotated_folder.mkdir(parents=True, exist_ok=True)
    debug_folder.mkdir(parents=True, exist_ok=True)

    for idx, frame_file in enumerate(frame_files):
        frame = cv2.imread(str(frame_file))
        frame = cv2.resize(frame, (width, height))
        frame = cv2.undistort(frame, mtx, dist)
        
        balls = detect_all_balls(frame)
        debug_frame = frame.copy()
        for ball_num, (x, y, r) in enumerate(balls):
            cv2.circle(debug_frame, (x, y), r, (0, 0, 255), 3)
            cv2.circle(debug_frame, (x, y), 5, (0, 255, 0), -1)
            positions.append([idx, ball_num, x, y, r])
            x1, y1 = max(x - r, 0), max(y - r, 0)
            x2, y2 = min(x + r, frame.shape[1]), min(y + r, frame.shape[0])
            cropped = frame[y1:y2, x1:x2]
            cropped_resized = cv2.resize(cropped, (512, 512), interpolation=cv2.INTER_AREA)
            cropped_path = cropped_folder / f"frame_{idx:04d}_ball{ball_num}.png"
            cv2.imwrite(str(cropped_path), cropped_resized)
            # # Optional marker detection
            # hsv = cv2.cvtColor(cropped_resized, cv2.COLOR_BGR2HSV)
            # mask = np.zeros(cropped.shape[:2], dtype=np.uint8)
            # cv2.circle(mask, (r, r), r, 255, -1)
            # mask_resized = cv2.resize(mask, (512, 512), interpolation=cv2.INTER_NEAREST)
            # colors = {'red': (0, 0, 255), 'blue': (255, 0, 0), 'green': (0, 255, 0)}
            # combined_annotated = cropped_resized.copy()
            # for color_name, bgr in colors.items():
            #     markers = detect_markers(hsv, color_name, mask_resized)
            #     for mx, my in markers:
            #         marker_records.append({
            #             "frame": idx,
            #             "ball_id": ball_num,
            #             "color": color_name,
            #             "marker_x": mx,
            #             "marker_y": my,
            #             "ball_cx": 256,
            #             "ball_cy": 256,
            #             "ball_r": 256
            #         })
            #         cv2.circle(combined_annotated, (mx, my), 5, bgr, -1)
            # annotated_path = annotated_folder / f"frame_{idx:04d}_ball{ball_num}_annotated.png"
            # cv2.imwrite(str(annotated_path), combined_annotated)
        debug_path = debug_folder / f"frame_{idx:04d}_all_balls.png"
        cv2.imwrite(str(debug_path), debug_frame)
    return np.array(positions), pd.DataFrame(marker_records)

# === 8. Main Multi-Ball Triangulation with Exhaustive Matching ===
def Localize3D_from_frames(
    frame_dirs, calib_files, calib_indices, width, height,
    start_frame=0, end_frame=None
):
    cam_params = []
    for i in range(2):
        K, dist, T, R, R_vec, w_calib, h_calib = getcameraparameters(calib_files[i], int(calib_indices[i]-1))
        mtx = AdjustedK(K, width, height, w_calib, h_calib)
        cam_params.append((mtx, dist, T, R, R_vec))

    P1 = cam_params[0][0] @ np.hstack([cam_params[0][3], cam_params[0][2]])
    P2 = cam_params[1][0] @ np.hstack([cam_params[1][3], cam_params[1][2]])

    positions_cam1, markers_cam1 = process_frames_for_camera(frame_dirs[0], cam_params[0][0], cam_params[0][1], width, height)
    positions_cam2, markers_cam2 = process_frames_for_camera(frame_dirs[1], cam_params[1][0], cam_params[1][1], width, height)

    df1 = pd.DataFrame(positions_cam1, columns=['frame', 'ball_id', 'x', 'y', 'r'])
    df2 = pd.DataFrame(positions_cam2, columns=['frame', 'ball_id', 'x', 'y', 'r'])
    # print(df1)
    # print(df2)
    if end_frame is None:
        end_frame = df1['frame'].max()
    frame_numbers = sorted(set(df1['frame']) & set(df2['frame']))
    frame_numbers = [f for f in frame_numbers if start_frame <= f <= end_frame]
    # print(frame_numbers)
    results_list = []
    for frame in frame_numbers:
        balls1 = df1[df1['frame'] == frame][['x','y']].to_numpy()
        balls2 = df2[df2['frame'] == frame][['x','y']].to_numpy()
        # print("balls1",balls1)
        # print("balls2",balls2)
        N1 = balls1.shape[0]
        N2 = balls2.shape[0]
        # print(N1)
        # print(N2)
        if N1 == N2 and N1 > 0:
            # ----------------------------
            # Y-sorting matching (insert here)
            idxs1 = np.argsort(balls1[:,1])  # sort cam1 balls by y
            idxs2 = np.argsort(balls2[:,1])  # sort cam2 balls by y
            # print(idxs1)
            # print(idxs2)
            balls1_sorted = balls1[idxs1]
            balls2_sorted = balls2[idxs2]
            pts_1 = balls1_sorted.T  # shape (2, N)
            pts_2 = balls2_sorted.T
            print(pts_1)
            print(pts_2)
            # Triangulate
            pts_4d = cv2.triangulatePoints(P1, P2, pts_1, pts_2)
            pts_3d = (pts_4d[:3] / pts_4d[3]).T  # Shape (N, 3)
            


            refined_3d = GaussNewtonRefinement(pts_3d, pts_1.T, pts_2.T, P1, P2)

            error_cam1 = ReProjectionError(refined_3d, pts_1.T, P1)
            error_cam2 = ReProjectionError(refined_3d, pts_2.T, P2)
            
            print(refined_3d)
            print(error_cam1)
            print(error_cam2)      
            for i in range(N1):
                # (expand dict fields as desired)
                results_list.append({
                    "frame": frame,
                    "ball_id": i,
                    "x_3d": refined_3d[i,0],
                    "y_3d": refined_3d[i,1],
                    "z_3d": refined_3d[i,2],
                    'reproj_error_cam1': error_cam1,
                    'reproj_error_cam2': error_cam2
                    # add more fields as needed
                })


    if len(results_list) == 0:
        print("No valid ball pairs found for triangulation.")
        return None, None, None

    df = pd.DataFrame(results_list)
    for frame_dir in frame_dirs:
        out_path = Path(frame_dir) / "3d_trajectory_refined.csv"
        df.to_csv(out_path, index=False)
    # markers_cam1.to_csv(Path(frame_dirs[0]) / "markers_cam1.csv", index=False)
    # markers_cam2.to_csv(Path(frame_dirs[1]) / "markers_cam2.csv", index=False)

    # error_stats = pd.DataFrame({
    #     'Camera': ['Camera 1', 'Camera 2'],
    #     'Mean Reprojection Error': [df['error_cam1'].mean(), df['error_cam2'].mean()],
    #     'Std Reprojection Error': [df['error_cam1'].std(), df['error_cam2'].std()]
    # })
    # for frame_dir in frame_dirs:
    #     error_stats.to_csv(Path(frame_dir) / "reprojection_error_stats.csv", index=False)

    print(f"✅ Refined trajectory saved to both {frame_dirs[0]} and {frame_dirs[1]}")
    print(f"✅ Marker positions saved to markers_cam1.csv and markers_cam2.csv in respective folders")
    # print(f"Camera 1 reprojection error: Mean={df['error_cam1'].mean():.3f}, Std={df['error_cam1'].std():.3f}")
    # print(f"Camera 2 reprojection error: Mean={df['error_cam2'].mean():.3f}, Std={df['error_cam2'].std():.3f}")

    return df, markers_cam1, markers_cam2

# === 9. Usage Example ===
if __name__ == "__main__":
    frame_dirs = ["D:/TT/CESSA/Validation_121125/20_20_C001H001S0001",
                  "D:/TT/CESSA/Validation_121125/20_20_C002H001S0001"]  # Folders with .jpg frames for each camera    
    # frame_dirs = ["D:/TT/Internship/sitara/Validation_setup/10_0/10cm_rightofrefpoint_C001H001S0001",
    #               "D:/TT/Internship/sitara/Validation_setup/10_0/10cm_rightofrefpoint_C002H001S0001"]  # Folders with .jpg frames for each camera
    # frame_dirs = ["D:/TT/Internship/sitara/Validation/positin002/position2_C001H001S0001",
    #               "D:/TT/Internship/sitara/Validation/positin002/position2_C002H001S0001"]  # Folders with .jpg frames for each camera
    # frame_dirs = ["D:/TT/Internship/sitara/Validation_Setup/Three/images_C001H001S0002",
    #               "D:/TT/Internship/sitara/Validation_Setup/Three/images_C002H001S0002"]  # Folders with .jpg frames for each camera
    calib_files = ["D:/TT/CESSA/Validation_121125/zcalib/C001calib.mat","D:/TT/CESSA/Validation_121125/zcalib/C002calib.mat"]

    calib_indices = [1, 1]
    width, height = 2048,1024 #2048, 2048

    # Restrict frames as needed:
    start_frame = 0
    end_frame = None

    trajectory_df, markers1, markers2 = Localize3D_from_frames(
        frame_dirs, calib_files, calib_indices, width, height,
        start_frame=start_frame, end_frame=end_frame
    )


[[1247 1208 1238 1175]
 [ 627  725  821  917]]
[[1214 1180 1219 1160]
 [ 609  705  801  897]]
[[ 269.59157092 -235.59175696  -81.27491767]
 [ 214.35414102  -87.49479179 -110.61061909]
 [ 267.86818623   53.88456906 -170.07877659]
 [ 175.47425836  200.5493339  -190.22769924]]
[0.37246954 0.83728841 0.38689919 0.3550987 ]
[0.37694402 0.84577687 0.39099134 0.35788626]
✅ Refined trajectory saved to both D:/TT/CESSA/Validation_121125/20_20_C001H001S0001 and D:/TT/CESSA/Validation_121125/20_20_C002H001S0001
✅ Marker positions saved to markers_cam1.csv and markers_cam2.csv in respective folders
